## SRP663430

**paper:** [10.64898/2026.01.27.702056](https://www.biorxiv.org/content/10.64898/2026.01.27.702056v1.full) - IAP retrotransposons contribute to the transcriptional diversity of the murine placenta, 2026

**date, curator:** 2026-04-23, Sara Carsanaro

**notes**
* no PMID yet
* removed cell culture/stem cells

### annotation summary

In [20]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Spleen,UBERON:0002106,spleen,perfect match
1,Liver,UBERON:0002107,liver,perfect match
2,Kidney,UBERON:0000082,adult mammalian kidney,perfect match
3,Brain,UBERON:0000955,brain,perfect match
4,Placenta,UBERON:0001987,placenta,perfect match


In [21]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,adult,UBERON:0000113,post-juvenile adult stage,perfect match
4,E14.5,UBERON:0018241,prime adult stage,perfect match
6,E14.5,MmusDv:0000136,prime adult stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP663430"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX31837288,SRP663430,Illumina NovaSeq X Plus,SRS27794394,UBERON:0002106,spleen,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458960,Spleen,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit","nan,full_length","ribo-minus,nan",,,Mus pahari spleen,SAMN54691244,,,,,,,,,,,23/04/2026,Total RNA was isolated using a commercial RNA extraction kit (Qiagen) and treated with DNase I (Ambion) to remove residual genomic DNA. 500 ng of total RNA was subjected to ribosomal RNA depletion using the NEBNext rRNA Depletion Kit v2. Strand-specific libraries were generated using the NEBNext Ultra II Directional RNA Library Prep Kit for Illumina according to the manufacturer's instructions.,,,,Spleen,,,,TRANSCRIPTOMIC,cDNA
1,SRX31837287,SRP663430,Illumina NovaSeq X Plus,SRS27794393,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458959,Liver,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit","nan,full_length","ribo-minus,nan",,,Mus pahari liver,SAMN54691245,,,,,,,,,,,23/04/2026,Total RNA was isolated using a commercial RNA extraction kit (Qiagen) and treated with DNase I (Ambion) to remove residual genomic DNA. 500 ng of total RNA was subjected to ribosomal RNA depletion using the NEBNext rRNA Depletion Kit v2. Strand-specific libraries were generated using the NEBNext Ultra II Directional RNA Library Prep Kit for Illumina according to the manufacturer's instructions.,,,,Liver,,,,TRANSCRIPTOMIC,cDNA
2,SRX31837286,SRP663430,Illumina NovaSeq X Plus,SRS27794392,UBERON:0000082,adult mammalian kidney,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458958,Kidney,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit","nan,full_length","ribo-minus,nan",,,Mus pahari kidney,SAMN54691246,,,,,,,,,,,23/04/2026,Total RNA was isolated using a commercial RNA extraction kit (Qiagen) and treated with DNase I (Ambion) to remove residual genomic DNA. 500 ng of total RNA was subjected to ribosomal RNA depletion using the NEBNext rRNA Depletion Kit v2. Strand-specific libraries were generated using the NEBNext Ultra II Directional RNA Library Prep Kit for Illumina according to the manufacturer's instructions.,,,,Kidney,,,,TRANSCRIPTOMIC,cDNA
3,SRX31837285,SRP663430,Illumina NovaSeq X Plus,SRS27794390,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458957,Brain,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit","nan,full_length","ribo-minus,nan",,,Mus pahari brain,SAMN54691247,,,,,,,,,,,23/04/2026,Total RNA was isolated using a commercial RNA extraction kit (Qiagen) and treated with DNase I (Ambion) to remove residual genomic DNA. 500 ng of total RNA was subjected to ribosomal RNA depletion using the NEBNext rRNA Depletion Kit v2. Strand-specific libraries were generated using the NEBNext Ultra II Directional RNA Library Prep Kit for Illumina according to the manufacturer's instructions.,,,,Brain,,,,TRANSCRIPTOMIC,cDNA
4,SRX31837284,SRP663430,Illumina Nova

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Brain' 'Kidney' 'Liver' 'Placenta' 'Spleen']


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [6]:
unique_sorted(library, "infoStage")

['E14.5' 'adult']


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [7]:
# making these variables because we use them again in the experiment file
my_protocol = 'NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'ribo-minus'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX31837288,SRP663430,Illumina NovaSeq X Plus,SRS27794394,UBERON:0002106,spleen,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458960,Spleen,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,ribo-minus,,,Mus pahari spleen,SAMN54691244,,,,,,,,,,,23/04/2026,Total RNA was isolated using a commercial RNA extraction kit (Qiagen) and treated with DNase I (Ambion) to remove residual genomic DNA. 500 ng of total RNA was subjected to ribosomal RNA depletion using the NEBNext rRNA Depletion Kit v2. Strand-specific libraries were generated using the NEBNext Ultra II Directional RNA Library Prep Kit for Illumina according to the manufacturer's instructions.,,,,Spleen,,,,TRANSCRIPTOMIC,cDNA
1,SRX31837287,SRP663430,Illumina NovaSeq X Plus,SRS27794393,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458959,Liver,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,ribo-minus,,,Mus pahari liver,SAMN54691245,,,,,,,,,,,23/04/2026,Total RNA was isolated using a commercial RNA extraction kit (Qiagen) and treated with DNase I (Ambion) to remove residual genomic DNA. 500 ng of total RNA was subjected to ribosomal RNA depletion using the NEBNext rRNA Depletion Kit v2. Strand-specific libraries were generated using the NEBNext Ultra II Directional RNA Library Prep Kit for Illumina according to the manufacturer's instructions.,,,,Liver,,,,TRANSCRIPTOMIC,cDNA
2,SRX31837286,SRP663430,Illumina NovaSeq X Plus,SRS27794392,UBERON:0000082,adult mammalian kidney,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458958,Kidney,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,ribo-minus,,,Mus pahari kidney,SAMN54691246,,,,,,,,,,,23/04/2026,Total RNA was isolated using a commercial RNA extraction kit (Qiagen) and treated with DNase I (Ambion) to remove residual genomic DNA. 500 ng of total RNA was subjected to ribosomal RNA depletion using the NEBNext rRNA Depletion Kit v2. Strand-specific libraries were generated using the NEBNext Ultra II Directional RNA Library Prep Kit for Illumina according to the manufacturer's instructions.,,,,Kidney,,,,TRANSCRIPTOMIC,cDNA
3,SRX31837285,SRP663430,Illumina NovaSeq X Plus,SRS27794390,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458957,Brain,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,ribo-minus,,,Mus pahari brain,SAMN54691247,,,,,,,,,,,23/04/2026,Total RNA was isolated using a commercial RNA extraction kit (Qiagen) and treated with DNase I (Ambion) to remove residual genomic DNA. 500 ng of total RNA was subjected to ribosomal RNA depletion using the NEBNext rRNA Depletion Kit v2. Strand-specific libraries were generated using the NEBNext Ultra II Directional RNA Library Prep Kit for Illumina according to the manufacturer's instructions.,,,,Brain,,,,TRANSCRIPTOMIC,cDNA
4,SRX31837284,SRP663430,Illumina NovaSeq X Plus,SRS27794391,UBERON:0001987,placenta,U

#### globin, replicates

In [8]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [9]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-23'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX31837288,SRP663430,Illumina NovaSeq X Plus,SRS27794394,UBERON:0002106,spleen,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458960,Spleen,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,ribo-minus,,,Mus pahari spleen,SAMN54691244,,,,,,,,,,SAC,2026-04-23,Total RNA was isolated using a commercial RNA extraction kit (Qiagen) and treated with DNase I (Ambion) to remove residual genomic DNA. 500 ng of total RNA was subjected to ribosomal RNA depletion using the NEBNext rRNA Depletion Kit v2. Strand-specific libraries were generated using the NEBNext Ultra II Directional RNA Library Prep Kit for Illumina according to the manufacturer's instructions.,,,,Spleen,,,,TRANSCRIPTOMIC,cDNA
1,SRX31837287,SRP663430,Illumina NovaSeq X Plus,SRS27794393,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458959,Liver,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,ribo-minus,,,Mus pahari liver,SAMN54691245,,,,,,,,,,SAC,2026-04-23,Total RNA was isolated using a commercial RNA extraction kit (Qiagen) and treated with DNase I (Ambion) to remove residual genomic DNA. 500 ng of total RNA was subjected to ribosomal RNA depletion using the NEBNext rRNA Depletion Kit v2. Strand-specific libraries were generated using the NEBNext Ultra II Directional RNA Library Prep Kit for Illumina according to the manufacturer's instructions.,,,,Liver,,,,TRANSCRIPTOMIC,cDNA
2,SRX31837286,SRP663430,Illumina NovaSeq X Plus,SRS27794392,UBERON:0000082,adult mammalian kidney,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458958,Kidney,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,ribo-minus,,,Mus pahari kidney,SAMN54691246,,,,,,,,,,SAC,2026-04-23,Total RNA was isolated using a commercial RNA extraction kit (Qiagen) and treated with DNase I (Ambion) to remove residual genomic DNA. 500 ng of total RNA was subjected to ribosomal RNA depletion using the NEBNext rRNA Depletion Kit v2. Strand-specific libraries were generated using the NEBNext Ultra II Directional RNA Library Prep Kit for Illumina according to the manufacturer's instructions.,,,,Kidney,,,,TRANSCRIPTOMIC,cDNA
3,SRX31837285,SRP663430,Illumina NovaSeq X Plus,SRS27794390,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458957,Brain,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,ribo-minus,,,Mus pahari brain,SAMN54691247,,,,,,,,,,SAC,2026-04-23,Total RNA was isolated using a commercial RNA extraction kit (Qiagen) and treated with DNase I (Ambion) to remove residual genomic DNA. 500 ng of total RNA was subjected to ribosomal RNA depletion using the NEBNext rRNA Depletion Kit v2. Strand-specific libraries were generated using the NEBNext Ultra II Directional RNA Library Prep Kit for Illumina according to the manufacturer's instructions.,,,,Brain,,,,TRANSCRIPTOMIC,cDNA
4,SRX31837284,SRP663430,Illumina NovaSeq X Plus,SRS27794391,UBERON:000198

#### comments

In [10]:
library.loc[:,'comment'] = 'https://www.biorxiv.org/content/10.64898/2026.01.27.702056v1.full'

#### save complete file with correct columns

In [11]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX31837288,SRP663430,Illumina NovaSeq X Plus,SRS27794394,UBERON:0002106,spleen,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458960,Spleen,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,ribo-minus,,,Mus pahari spleen,SAMN54691244,,,,,,,https://www.biorxiv.org/content/10.64898/2026.01.27.702056v1.full,,,SAC,2026-04-23
1,SRX31837287,SRP663430,Illumina NovaSeq X Plus,SRS27794393,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458959,Liver,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,ribo-minus,,,Mus pahari liver,SAMN54691245,,,,,,,https://www.biorxiv.org/content/10.64898/2026.01.27.702056v1.full,,,SAC,2026-04-23
2,SRX31837286,SRP663430,Illumina NovaSeq X Plus,SRS27794392,UBERON:0000082,adult mammalian kidney,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458958,Kidney,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,ribo-minus,,,Mus pahari kidney,SAMN54691246,,,,,,,https://www.biorxiv.org/content/10.64898/2026.01.27.702056v1.full,,,SAC,2026-04-23
3,SRX31837285,SRP663430,Illumina NovaSeq X Plus,SRS27794390,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458957,Brain,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,ribo-minus,,,Mus pahari brain,SAMN54691247,,,,,,,https://www.biorxiv.org/content/10.64898/2026.01.27.702056v1.full,,,SAC,2026-04-23
4,SRX31837284,SRP663430,Illumina NovaSeq X Plus,SRS27794391,UBERON:0001987,placenta,UBERON:0018241,prime adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458956,Placenta,E14.5,perfect match,not documented,perfect match,F,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,ribo-minus,,,"Mus pahari placenta, rep 2",SAMN54691248,14.5,embryonic day,,,,,https://www.biorxiv.org/content/10.64898/2026.01.27.702056v1.full,,,SAC,2026-04-23
5,SRX31837283,SRP663430,Illumina NovaSeq X Plus,SRS27794389,UBERON:0001987,placenta,UBERON:0018241,prime adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458955,Placenta,E14.5,perfect match,not documented,perfect match,F,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,ribo-minus,,,"Mus pahari placenta, rep 1",SAMN54691249,14.5,embryonic day,,,,,https://www.biorxiv.org/content/10.64898/2026.01.27.702056v1.full,,,SAC,2026-04-23
6,SRX31837282,SRP663430,Illumina NovaSeq X Plus,SRS27794388,UBERON:0001987,placenta,MmusDv:0000136,prime adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM9458954,Placenta,E14.5,perfect match,not documented,perfect match,F,,,10090,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,ribo-minus,,,"Mus musculus placenta, rep 3",SAMN54691250,14.5,embryonic day,,,,,https://www.biorxiv.org/content/10.64898/2026.01.27.702056v1.full,,,SAC,2026-04-23
7,SRX31837281,SRP663430,Illumina NovaSeq X Plus,SRS27794387,UBERON:0001987,placenta,MmusDv:0000136,prime adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM945895

### experiment annotations

In [12]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP663430,IAP retrotransposons contribute to the transcriptional diversity of the murine placenta [RNA-Seq],"Transposable elements (TEs) have made important contributions to the evolution of the placenta, and are argued to have played a role in the wide inter-species diversification of this critical developmental organ. Co-option of TEs by host genomes has led to the genesis of important placental genes, as well as trophoblast-specific gene regulatory elements. In mice, past work has demonstrated how multiple species-specific TE subfamilies are used as transcriptional enhancers in trophoblast stem cells. However, the involvement of TEs in the regulation of mouse placental gene expression in vivo remains unclear. Here, we characterised the TE regulatory and transcriptional landscape in mouse placenta and gauged their evolutionary dynamics through a comparative approach. We found that overall, TE cis-regulatory activity is greatly diminished in differentiated mouse trophoblast when compared to their stem cell counterpart. On the other hand, evolutionarily young IAP elements are highly expressed in the placenta and create several alternative, placenta-specific transcriptional start sites for protein-coding genes. Placenta-expressed IAP elements are genetically polymorphic between mouse strains and drive species-specific expression of associated genes. These putative co-option events are therefore recent and may represent a prime example of how TE activity can drive fast placental evolution. Overall design: RNA-seq profiling of Mus musculus placenta and trophoblast stem cells, and Mus pahari placenta and selected adult tissues. CRISPR interference of IAP endogenous retroviruses in mouse trophoblast stem cells.",SRA,,,,"NEBNext rRNA Depletion Kit, NEBNext Ultra II Directional RNA Library Prep Kit","nan, full_length",,PRJNA1403818,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [13]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

9

In [14]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP663430,IAP retrotransposons contribute to the transcriptional diversity of the murine placenta [RNA-Seq],"Transposable elements (TEs) have made important contributions to the evolution of the placenta, and are argued to have played a role in the wide inter-species diversification of this critical developmental organ. Co-option of TEs by host genomes has led to the genesis of important placental genes, as well as trophoblast-specific gene regulatory elements. In mice, past work has demonstrated how multiple species-specific TE subfamilies are used as transcriptional enhancers in trophoblast stem cells. However, the involvement of TEs in the regulation of mouse placental gene expression in vivo remains unclear. Here, we characterised the TE regulatory and transcriptional landscape in mouse placenta and gauged their evolutionary dynamics through a comparative approach. We found that overall, TE cis-regulatory activity is greatly diminished in differentiated mouse trophoblast when compared to their stem cell counterpart. On the other hand, evolutionarily young IAP elements are highly expressed in the placenta and create several alternative, placenta-specific transcriptional start sites for protein-coding genes. Placenta-expressed IAP elements are genetically polymorphic between mouse strains and drive species-specific expression of associated genes. These putative co-option events are therefore recent and may represent a prime example of how TE activity can drive fast placental evolution. Overall design: RNA-seq profiling of Mus musculus placenta and trophoblast stem cells, and Mus pahari placenta and selected adult tissues. CRISPR interference of IAP endogenous retroviruses in mouse trophoblast stem cells.",SRA,partial,Bgee 1K,9,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,,PRJNA1403818,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [15]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
#experiment.loc[:,'PMID'] = ''
experiment.loc[:,'reference_url'] = 'https://www.biorxiv.org/content/10.64898/2026.01.27.702056v1.full'
experiment.loc[:,'DOI'] = '10.64898/2026.01.27.702056'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP663430,IAP retrotransposons contribute to the transcriptional diversity of the murine placenta [RNA-Seq],"Transposable elements (TEs) have made important contributions to the evolution of the placenta, and are argued to have played a role in the wide inter-species diversification of this critical developmental organ. Co-option of TEs by host genomes has led to the genesis of important placental genes, as well as trophoblast-specific gene regulatory elements. In mice, past work has demonstrated how multiple species-specific TE subfamilies are used as transcriptional enhancers in trophoblast stem cells. However, the involvement of TEs in the regulation of mouse placental gene expression in vivo remains unclear. Here, we characterised the TE regulatory and transcriptional landscape in mouse placenta and gauged their evolutionary dynamics through a comparative approach. We found that overall, TE cis-regulatory activity is greatly diminished in differentiated mouse trophoblast when compared to their stem cell counterpart. On the other hand, evolutionarily young IAP elements are highly expressed in the placenta and create several alternative, placenta-specific transcriptional start sites for protein-coding genes. Placenta-expressed IAP elements are genetically polymorphic between mouse strains and drive species-specific expression of associated genes. These putative co-option events are therefore recent and may represent a prime example of how TE activity can drive fast placental evolution. Overall design: RNA-seq profiling of Mus musculus placenta and trophoblast stem cells, and Mus pahari placenta and selected adult tissues. CRISPR interference of IAP endogenous retroviruses in mouse trophoblast stem cells.",SRA,partial,Bgee 1K,9,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,,PRJNA1403818,,https://www.biorxiv.org/content/10.64898/2026.01.27.702056v1.full,10.64898/2026.01.27.702056,,


#### comments

In [16]:
experiment.loc[:,'comment'] = 'removed cell culture libraries. no PMID yet, still preprint'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP663430,IAP retrotransposons contribute to the transcriptional diversity of the murine placenta [RNA-Seq],"Transposable elements (TEs) have made important contributions to the evolution of the placenta, and are argued to have played a role in the wide inter-species diversification of this critical developmental organ. Co-option of TEs by host genomes has led to the genesis of important placental genes, as well as trophoblast-specific gene regulatory elements. In mice, past work has demonstrated how multiple species-specific TE subfamilies are used as transcriptional enhancers in trophoblast stem cells. However, the involvement of TEs in the regulation of mouse placental gene expression in vivo remains unclear. Here, we characterised the TE regulatory and transcriptional landscape in mouse placenta and gauged their evolutionary dynamics through a comparative approach. We found that overall, TE cis-regulatory activity is greatly diminished in differentiated mouse trophoblast when compared to their stem cell counterpart. On the other hand, evolutionarily young IAP elements are highly expressed in the placenta and create several alternative, placenta-specific transcriptional start sites for protein-coding genes. Placenta-expressed IAP elements are genetically polymorphic between mouse strains and drive species-specific expression of associated genes. These putative co-option events are therefore recent and may represent a prime example of how TE activity can drive fast placental evolution. Overall design: RNA-seq profiling of Mus musculus placenta and trophoblast stem cells, and Mus pahari placenta and selected adult tissues. CRISPR interference of IAP endogenous retroviruses in mouse trophoblast stem cells.",SRA,partial,Bgee 1K,9,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Directional RNA Library Prep Kit",full_length,,PRJNA1403818,,https://www.biorxiv.org/content/10.64898/2026.01.27.702056v1.full,10.64898/2026.01.27.702056,,"removed cell culture libraries. no PMID yet, still preprint"


#### save complete file

In [17]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [18]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [19]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [22]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [23]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
62447,SRX11086097,SRP323114,NextSeq 500,SRS9151678,CL:0000018,spermatid,UBERON:0018241,prime adult stage,,RS,93 dpp,perfect match,not documented,other,M,PAHARI/EiJ,,10093,Agilent SureSelect,full_length,polyA,,38,PAHNew-4M-RS,SAMN19597784,93,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23
62448,SRX11086096,SRP323114,NextSeq 500,SRS9151678,CL:0000018,spermatid,UBERON:0018241,prime adult stage,,RS,93 dpp,perfect match,not documented,other,M,PAHARI/EiJ,,10093,Agilent SureSelect,full_length,polyA,,38,PAHNew-4M-RS,SAMN19597784,93,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23
62449,SRX31837288,SRP663430,Illumina NovaSeq X Plus,SRS27794394,UBERON:0002106,spleen,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Spleen,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Di...",full_length,ribo-minus,,,Mus pahari spleen,SAMN54691244,,,,,,,https://www.biorxiv.org/content/10.64898/2026....,,,SAC,2026-04-23
62450,SRX31837287,SRP663430,Illumina NovaSeq X Plus,SRS27794393,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Liver,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Di...",full_length,ribo-minus,,,Mus pahari liver,SAMN54691245,,,,,,,https://www.biorxiv.org/content/10.64898/2026....,,,SAC,2026-04-23
62451,SRX31837286,SRP663430,Illumina NovaSeq X Plus,SRS27794392,UBERON:0000082,adult mammalian kidney,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Kidney,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Di...",full_length,ribo-minus,,,Mus pahari kidney,SAMN54691246,,,,,,,https://www.biorxiv.org/content/10.64898/2026....,,,SAC,2026-04-23
62452,SRX31837285,SRP663430,Illumina NovaSeq X Plus,SRS27794390,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Brain,adult,perfect match,not documented,perfect match,NA,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Di...",full_length,ribo-minus,,,Mus pahari brain,SAMN54691247,,,,,,,https://www.biorxiv.org/content/10.64898/2026....,,,SAC,2026-04-23
62453,SRX31837284,SRP663430,Illumina NovaSeq X Plus,SRS27794391,UBERON:0001987,placenta,UBERON:0018241,prime adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Placenta,E14.5,perfect match,not documented,perfect match,F,,,10093,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Di...",full_length,ribo-minus,,,"Mus pahari placenta, rep 2",SAMN54691248,14.5,embryonic day,,,,,https://www.biorxiv.org/content/10.64898/2026....,,,SAC,2026-04-23


In [24]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1204,ERP139798,Convergent adaptation to xeric environments in...,Our study aims at better understanding the mol...,SRA,total,Bgee 1K,57,TruSeq RNA Library Prep Kit,full_length,,PRJEB54931,41565469,https://pmc.ncbi.nlm.nih.gov/articles/PMC12951...,10.1101/gr.280089.124,,
1205,SRP323114,RNAseq of FACS sorted testes from Mus wild-der...,We investigated the evolution of gene expressi...,SRA,total,Bgee 1K,139,Agilent SureSelect,full_length,,PRJNA735780,35099536,https://pmc.ncbi.nlm.nih.gov/articles/PMC8844503/,10.1093/molbev/msac023,,
1206,SRP663430,IAP retrotransposons contribute to the transcr...,Transposable elements (TEs) have made importan...,SRA,partial,Bgee 1K,9,"NEBNext rRNA Depletion Kit,NEBNext Ultra II Di...",full_length,,PRJNA1403818,,https://www.biorxiv.org/content/10.64898/2026....,10.64898/2026.01.27.702056,,"removed cell culture libraries. no PMID yet, s..."


### add annotations to git

In [25]:
! git pull

Already up to date.


In [26]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [27]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../SRP000401/
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [28]:
! git add $git_experiment_path $git_library_path

In [29]:
! git commit -m $commit_message_exp

[develop 4f0e7fd] adding annotated bulk experiment SRP663430
 2 files changed, 10 insertions(+)


In [30]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 3.03 KiB | 1.01 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   be3dec6..4f0e7fd  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push